# Selection and sampling sensitivity
Run after notebook 01. This notebook reports attrition and compares equal-eddy with day-weighted centrelines using the saved members. It does not silently generate additional scientific groups. For depth/threshold/normalisation sensitivity, change notebook 00 controls and execute 00–01 again; each setting has a distinct run hash.

Recommended initial alternatives: a shallower matched depth range, slope coherence 0.3/0.7, dominance factors 1.5/3, and kilometre coordinates. Thresholds should not be selected to maximise the desired tilt signal. High/low Rossby and stratification composites are a subsequent extension once these baseline diagnostics are reviewed.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
HERE = Path.cwd().resolve()
ANALYSIS = next((p for p in (HERE, *HERE.parents) if (p/'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or a subdirectory')
WORK = ANALYSIS/'esp_population_composites'
for p in (ANALYSIS, WORK):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
import seacofs_tilt_tools as tilt
import population_tools as pop
pd.set_option('display.max_columns', 60)


In [2]:
OUTPUT_ROOT = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/esp_population_composites')
run = Path((OUTPUT_ROOT/'latest_run.txt').read_text().strip())
audit = pd.read_parquet(run/'audit.parquet')
manifest = pd.read_csv(run/'members/members.csv')
config = json.loads((run/'provenance.json').read_text())['config']
rows=[]
for group,part in manifest.groupby('group'):
    member = []
    for file in part.file:
        with np.load(run/'members'/file) as m: member.append(m['centres'])
    member=np.stack(member)
    equal=member.mean(axis=0)
    day=np.average(member, axis=0, weights=part.days)
    for k,depth in enumerate(config['depths']):
        rows.append(dict(group=group, depth=depth, eddies=len(part), days=part.days.sum(),
                         eddy_equal_axis1=equal[k,0], day_weighted_axis1=day[k,0],
                         eddy_equal_axis2=equal[k,1], day_weighted_axis2=day[k,1],
                         median_member_offset=np.median(np.linalg.norm(member[:,k],axis=1)),
                         norm_mean_offset=np.linalg.norm(equal[k])))
comparison=pd.DataFrame(rows)
display(comparison)
comparison.to_csv(run/'weighting_sensitivity.csv',index=False)
display(audit.groupby(['Cyc','regime','selected'],dropna=False).agg(days=('Day','size'),eddies=('Eddy','nunique')))


,group,depth,eddies,days,eddy_equal_axis1,day_weighted_axis1,eddy_equal_axis2,day_weighted_axis2,median_member_offset,norm_mean_offset
0,AE_Planetary,0.000000,382,3926,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,AE_Planetary,181.264793,382,3926,0.022170,0.013476,-0.002628,-0.008140,0.070765,0.022325
2,AE_Planetary,515.416489,382,3926,0.057413,0.027159,-0.008030,-0.028646,0.131197,0.057971
3,AE_Topographic,0.000000,1190,19353,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,AE_Topographic,181.264793,1190,19353,-0.034162,-0.018505,0.001866,0.005235,0.085598,0.034212
5,AE_Topographic,515.416489,1190,19353,-0.057966,-0.049224,-0.012005,-0.008397,0.215362,0.059196
6,CE_Planetary,0.000000,317,1528,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,CE_Planetary,181.264793,317,1528,-0.047724,-0.038582,-0.018810,-0.007160,0.085873,0.051297
8,CE_Planetary,515.416489,317,1528,-0.081781,-0.079649,-0.006771,0.002805,0.139966,0.082061
9,CE_Topographic,0.000000,1407,29000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


days  eddies
Cyc regime      selected               
AE  Mixed       False     17884     985
    Planetary   False       538     226
                True       3926     382
    Topographic False     19852    1362
                True      19353    1190
    Unknown     False      3399    1005
CE  Mixed       False     10509     858
    Planetary   False       125      86
                True       1528     317
    Topographic False     17828    1402
                True      29000    1407
    Unknown     False      3484    1063

## Required scientific checks before interpretation
- Compare maps and latitude/season distributions, not only the composite arrows.
- Inspect real-data reconstruction skill for a stratified set of members using `case_studies/eddy_cross_sections_3d/eddy_sections_and_esp.ipynb`.
- Check complete-profile selection against the shallower run.
- Examine broad individual-centre distributions when the composite looks upright.
- Treat small populations as exploratory; bootstrap draws do not create independent information.
- Interpolation provenance is unresolved in the processed surface output. An observed-only sensitivity requires upstream provenance with a stable ID mapping.
- Core N² is not background stratification. The current four groups require no N² cache and do not test a stratification mechanism.
